<a href="https://colab.research.google.com/github/yeshaa23/ZARA-AppsReview-SentimentAnalysis/blob/main/Scrapping_link_Pertamina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install dan import Library

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install newspaper3k Sastrawi Stopword nltk wordcloud lxml_html_clean -q


In [3]:
import pandas as pd
import numpy as np
import newspaper
from newspaper import Config
from urllib.parse import urlparse
import warnings
warnings.filterwarnings("ignore")

## UPLOAD FILE CSV

In [4]:
df = pd.read_csv('/content/drive/MyDrive/ProjectA-PBA/Link_Scraping_Pertamina.csv')

In [5]:
print("Jumlah data awal:", len(df))
display(df.head())

# Ambil semua link
list_of_urls = df['link'].dropna().tolist()

Jumlah data awal: 100


,link,tag
0,https://www.esdm.go.id/id/media-center/arsip-b...,ekonomi
1,https://ylki.or.id/siaran-pers-ylki-mendesak-p...,hukum
2,https://money.kompas.com/read/2017/07/06/14535...,bbm
3,https://otomotif.kompas.com/read/2017/12/14/15...,bisnis
4,https://money.kompas.com/read/2017/07/27/17565...,bisnis


## Scrapping

In [6]:
# config scraping
config = Config()
config.request_timeout = 30
config.browser_user_agent = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/122.0.0.0 Safari/537.36"
)

article_details_list = []

In [7]:
# looping scraping
for i, row in df.iterrows():
    url = row['link']
    tag = row['tag'] if 'tag' in df.columns else None

    try:
        article = newspaper.Article(url=url, language='id', config=config)
        article.download()
        article.parse()

        # summary kadang kosong kalau belum nlp()
        try:
            article.nlp()
            summary = article.summary
        except:
            summary = None

        text = article.text
        parsed_url = urlparse(url)
        source = parsed_url.netloc.replace('www.', '')

        # fallback summary kalau kosong
        if not summary or str(summary).strip() == "":
            meta_data = article.meta_data if hasattr(article, 'meta_data') else {}
            summary = meta_data.get('description', None)

        article_detail = {
            "title": str(article.title) if article.title else None,
            "authors": ", ".join(article.authors) if article.authors else None,
            "source": source,
            "published_date": str(article.publish_date) if article.publish_date else None,
            "summary": summary,
            "content": text if text else None,
            "url": url,
            "tag": tag
        }

        article_details_list.append(article_detail)

    except newspaper.article.ArticleException as e:
        print(f"Failed to process URL: {url}\nError: {e}\n")
    except Exception as e:
        print(f"An unexpected error occurred for URL: {url}\nError: {e}\n")

Failed to process URL: https://money.kompas.com/read/2016/03/18/141525826/Gaji.Manajer.Engineering.Chevron.Rp.51.25.Juta.di.Pertamina.Rp.13.3.Juta.per.Bulan
Error: Article `download()` failed with ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer')) on URL https://money.kompas.com/read/2016/03/18/141525826/Gaji.Manajer.Engineering.Chevron.Rp.51.25.Juta.di.Pertamina.Rp.13.3.Juta.per.Bulan

Failed to process URL: https://otomotif.kompas.com/read/2017/12/13/162200515/setelah-lamborghini-pertamax-turbo-incar-mercedes-g-class
Error: Article `download()` failed with ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer')) on URL https://otomotif.kompas.com/read/2017/12/13/162200515/setelah-lamborghini-pertamax-turbo-incar-mercedes-g-class

Failed to process URL: https://otomotif.kompas.com/read/2017/10/18/150200515/sebenarnya-boleh-tidak-isi-bensin-pakai-jeriken-
Error: Article `download()` failed with ('Connection aborted.', ConnectionResetErr

## Dataframe hasil

In [8]:
df_pertamina = pd.DataFrame(article_details_list)

print("Jumlah artikel yang berhasil discrape:", len(df_pertamina))
display(df_pertamina.head())

Jumlah artikel yang berhasil discrape: 95


,title,authors,source,published_date,summary,content,url,tag
0,"Transparan dalam Pengadaan, ISC Bikin Pertamin...",None,esdm.go.id,None,Kementerian Energi dan Sumber Daya Mineral - R...,"Transparan dalam Pengadaan, ISC Bikin Pertamin...",https://www.esdm.go.id/id/media-center/arsip-b...,None
1,Siaran Pers YLKI : Mendesak PT Pertamina untuk...,None,ylki.or.id,None,None,Kejadian kecurangan takaran pada SPBU yang ter...,https://ylki.or.id/siaran-pers-ylki-mendesak-p...,None
2,Pertamina Kembangkan Program Non-Tunai Pembeli...,"Kompas Cyber Media, Kontributor Pontianak, Yoh...",money.kompas.com,2017-07-06 00:00:00,"Di Pontianak mulai tanggal 1 Januari 2018, unt...","PONTIANAK, KOMPAS.com - PT Pertamina mensosial...",https://money.kompas.com/read/2017/07/06/14535...,None
3,"Pertamina Tak Masalahkan Nama Pertamini, tapi...","Kompas Cyber Media, Alsadad Rudi, Febri Ardani...",otomotif.kompas.com,2017-12-14 00:00:00,Pertamini bukan bagian dari Pertamina. Namun P...,"Nusa Dua, KompasOtomotif - Keberadaan kios-kio...",https://otomotif.kompas.com/read/2017/12/14/15...,None
4,Pertamina Patra Niaga Akan Bangun SPBU Skala U...,"Kompas Cyber Media, Moh. Nadlir, Pertamina",money.kompas.com,2017-07-27 00:00:00,Nilai investasi pembangunan SPBU UKM itu ditak...,"JAKARTA, KOMPAS.com - PT Pertamina Patra Niaga...",https://money.kompas.com/read/2017/07/27/17565...,None


In [9]:
# Cek jumlah published_date yang none
none_count = df_pertamina['published_date'].astype(str).str.contains('None', case=False, na=False).sum()
print(f"Jumlah 'None' pada kolom published_date: {none_count}")

Jumlah 'None' pada kolom published_date: 14


In [10]:
# Hapus row yang published_date = None
df_pertamina = df_pertamina[df_pertamina['published_date'].astype(str) != 'None']
df_pertamina = df_pertamina[df_pertamina['published_date'].notna()]

none_count_after = df_pertamina['published_date'].astype(str).str.contains('None', case=False, na=False).sum()
print(f"Jumlah 'None' setelah dibersihkan: {none_count_after}")

Jumlah 'None' setelah dibersihkan: 0


In [12]:
# reset index
df_pertamina = df_pertamina.reset_index(drop=True)

## Simpan hasil

In [13]:
output_path = '/content/drive/MyDrive/ProjectA-PBA/Scraping_Pertamina.csv'
df_pertamina.to_csv(output_path, index=False)

print("File berhasil disimpan di:", output_path)

File berhasil disimpan di: /content/drive/MyDrive/ProjectA-PBA/Scraping_Pertamina.csv


In [14]:
# lihat ringkasan hasil
print("Ukuran akhir data:", df_pertamina.shape)
display(df_pertamina[['title', 'source', 'published_date', 'tag', 'url']].head(10))

Ukuran akhir data: (81, 8)


,title,source,published_date,tag,url
0,Pertamina Kembangkan Program Non-Tunai Pembeli...,money.kompas.com,2017-07-06 00:00:00,None,https://money.kompas.com/read/2017/07/06/14535...
1,"Pertamina Tak Masalahkan Nama Pertamini, tapi...",otomotif.kompas.com,2017-12-14 00:00:00,None,https://otomotif.kompas.com/read/2017/12/14/15...
2,Pertamina Patra Niaga Akan Bangun SPBU Skala U...,money.kompas.com,2017-07-27 00:00:00,None,https://money.kompas.com/read/2017/07/27/17565...
3,Cara Mudah Lacak Pertamax Turbo dan Dex,otomotif.kompas.com,2017-12-15 00:00:00,None,https://otomotif.kompas.com/read/2017/12/15/12...
4,"Mulai Hari ini, Harga BBM Pertamina Naik",otomotif.kompas.com,2017-01-05 00:00:00,None,https://otomotif.kompas.com/read/2017/01/05/13...
5,"Bos Pertamina Bilang ""Sponsorship"" untuk Rio H...",money.kompas.com,2016-02-22 00:00:00,None,https://money.kompas.com/read/2016/02/22/22073...
6,Dirut dan Wadirut Pertamina Resmi Dicopot,money.kompas.com,2017-02-03 00:00:00,None,https://money.kompas.com/read/2017/02/03/11424...
7,Tanggapan Pertamina Soal SPBU Curang di Kemayoran,otomotif.kompas.com,2017-05-31 00:00:00,None,https://otomotif.kompas.com/read/2017/05/31/13...
8,Rincian Harga BBM Kemasan Kaleng Pertamina,otomotif.kompas.com,2016-07-05 00:00:00,None,https://otomotif.kompas.com/read/2016/07/05/07...
9,Pertamina Jamin Stok BBM Jelang Mudik 2017,otomotif.kompas.com,2017-05-31 00:00:00,None,https://otomotif.kompas.com/read/2017/05/31/04...
